# 9.13 缺資訊，應該輸出低信心嗎？


「請幫我查進度」少了要查哪一筆申請。這時我們其實很確定一件事：**需要先澄清**。所以缺資訊時，模型不必把每個動作都報成差不多的機率、表現得拿不定主意；它可以很有把握地選「先詢問」。要做到這點，得先分清楚問題問的是「使用者想做什麼」，還是「現在的資訊足以執行嗎」。

沿用同一個客服世界，我們另外問一個 yes／no 問題：「已提供此流程要求的申請編號嗎？」模型輸出 `P(yes | state, question)`，讀作「給定狀態與問題時答 yes 的機率」；no 的機率是1減掉它。這就是[9.11](https://birdhackor.github.io/tiny-perceptron-vlm/9.11.html)的 Noul 型別。第19章的機率判斷器會明列本世界的必要欄位，讓讀者能自己核對標籤，並不假設機率等於真實正確率。

| 材料 | 流程規則 | 資訊充分標籤 | 下一步標籤 |
| --- | --- | --- | --- |
| 查卡片進度，申請編號K17 | 此示範流程查進度要申請編號 | yes | 查卡片進度 |
| 查卡片進度，未給編號 | 同上 | no | 詢問申請編號 |
| 查地址變更規則 | 查規則不需個人編號 | yes | 查地址變更規則 |
| 重設密碼 | 本世界沒有這個流程 | 不適用 | 轉給真人客服 |

「下一步」的候選和9.11不同：問題換了，候選也跟著重新定義，所以這裡多了「詢問申請編號」這種澄清動作；資訊齊全時，下一步就是執行該任務。最後一列的「重設密碼」不在本世界的流程裡，「已提供此流程要求的申請編號嗎？」對它沒有意義，yes、no 都不對，所以標成不適用，再照9.11的規則轉給真人客服。表中的標籤可以由規則算出，不必靠人逐列猜：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
needs = {"查卡片進度": ["申請編號"], "查地址變更規則": []}  # 教學世界自訂：任務 → 還需要的欄位
materials = [
    {"任務": "查卡片進度", "申請編號": "K17"},
    {"任務": "查卡片進度"},
    {"任務": "查地址變更規則"},
    {"任務": "重設密碼"},
]
for m in materials:
    task = m.get("任務")
    if task is None:
        enough, step = "no", "詢問任務"
    elif task not in needs:
        enough, step = "不適用", "轉給真人客服"
    else:
        missing = [field for field in needs[task] if field not in m]
        enough = "yes" if len(missing) == 0 else "no"
        step = task if len(missing) == 0 else "詢問" + missing[0]
    print(m, "→ 資訊充分", enough, "下一步", step)

`needs`替每個任務記下還需要哪些欄位；每份材料是一個字典。迴圈裡的`m.get("任務")`取出任務；和`m["任務"]`不同，鍵不存在時它回傳`None`而不會報錯。接著的`if`、`elif`、`else`分三種情況，`elif`是「否則，如果」：前面的條件不成立時才檢查它。沒說任務，就是資訊不足，先詢問任務；任務不在`needs`裡（`task not in needs`問的是這個字典有沒有這個鍵），本世界沒有這個流程，資訊充分的問題不適用，照9.11的規則轉給真人客服；其餘情況才逐一檢查欄位。

`[field for field in needs[task] if field not in m]`是[7.12](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)用過的寫法，只留下材料裡沒有的欄位。缺的清單長度是0，就是資訊充分，下一步直接執行任務；否則下一步是詢問第一個缺的欄位。四行輸出正好重現上表。

Noul 只回答 yes 的機率，沒有「不適用」這個選項。所以實際系統要先用9.11那種含「以上皆不適合」的 Choice 問題擋下本世界沒有的任務，通過後才採用資訊是否充分的答案。第19章的 Auto 也是這樣：先看題目是否屬於支援的任務，不支援就直接回報 `unsupported`（不支援），不再看資訊是否足夠（[19.17](https://birdhackor.github.io/tiny-perceptron-vlm/19.17.html)，讀到第19章再看）。

這裡的規則是教學世界事先訂定，不是現實銀行共通規定。它使「資訊充分」有可檢查的意思：不是材料越長越充分，而是有沒有滿足**當前動作需要的條件**。同一材料換任務，答案可以改變。

有了可核對的標籤，模型要學的就是對第二份材料報出很低的 yes 機率。假設訓練好的模型報 P(yes)=0.03，看看這時的信心有多高：


In [ ]:
p_yes = 0.03  # 假設訓練好的模型對第二份材料報的 yes 機率
print("P(yes)", p_yes, "P(no)", round(1 - p_yes, 2), "信心", max(p_yes, 1 - p_yes))

no 的機率是1−0.03=0.97；依[9.9](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)的定義，信心是兩者中較大的0.97。缺資訊的這一題，模型其實很有把握，把握的內容正是「資訊不足」。這就回答了本節的標題：缺資訊不必用低信心表達，可以做成一個能被很有把握回答的問題。

準備標籤時，先用規則求出正確答案，再把缺欄位、無關編號及多餘文字的材料也納入。例如字串K17只是格式像編號，不代表一定屬於當前使用者：上面的程式只檢查欄位在不在，沒有核對編號真假；若世界沒有教驗證身分，就不能聲稱已驗身分。未知事實也不可由產生資料的程式暗中填答案。有時會用**教師模型**（另一個已訓練好、替新模型提供答案或機率的模型；第18章才詳細講，見[18.1](https://birdhackor.github.io/tiny-perceptron-vlm/18.1.html)）產生機率當訓練目標，但這只是一種訓練訊號，需要抽查並對照獨立的正確答案，不能自動升格成事實。

部署規則可以先看資訊充分的機率，再決定是否澄清，但門檻要用另一份沒拿來訓練的資料選，[9.14](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)稱它為決策資料。若輸入是未教過的服務，模型仍可能很肯定；下一節會讓機率與實際成敗對照，而不是信任小數點的精確外觀。

練習：在第一段程式的`materials`最後加一筆`{"申請編號": "K17"}`，也就是給了編號卻沒說要辦什麼。先預測新的一行是`資訊充分 no 下一步 詢問任務`，不能直接查卡片，再執行核對。資訊是否足夠要依明確的問題與流程判斷；問任務類型和問編號是否存在，是兩個不同的標籤。
